In [1]:
!pip install transformers datasets torch scikit-learn

  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 11.9 MB/s  0:00:07m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 10.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.1/8.1 MB 14.5 MB/s  0:00:00 eta 0:00:01
Using cached networkx-3.6.1-py3-none-any.whl (2.1 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.3/20.3 MB 16.2 MB/s  0:00:01m0:00:0100:01
Using cached sympy-1.14.0-py3-none-any.whl (6.3 MB)
Using cached mpmath-1.3.0-py3-none-any.whl (536 kB)
Using cached threadpoolctl-3.6.0-py3-none-any.whl (18 kB)
  Attempting uninstall: setuptoolsm━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/9 [sympy]
    Found existing installation: setuptools 82.0.1━━━━━━━━━━━━ 2/9 [sympy]
    Uninstalling setuptools-82.0.

In [ ]:
from datasets import load_dataset
import pandas as pd

imdb = load_dataset("imdb")

train_df = pd.DataFrame(imdb["train"])
test_df = pd.DataFrame(imdb["test"])

train_texts = train_df["text"]
train_labels = train_df["label"]

test_texts = test_df["text"]
test_labels = test_df["label"]

print(y_test.value_counts())

In [45]:
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

train_encodings = tokenizer(
    list(train_texts),
    truncation=True,
    padding=True,
    max_length=200
)

test_encodings = tokenizer(
    list(test_texts),
    truncation=True,
    padding=True,
    max_length=200
)

In [46]:
class IMDbDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels.iloc[idx])
        return item

    def __len__(self):
        return len(self.labels)

In [47]:
train_dataset = IMDbDataset(train_encodings, train_labels)
test_dataset = IMDbDataset(test_encodings, test_labels)

In [48]:
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=8)

In [24]:
model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)

Loading weights: 100%|██████████████████████| 199/199 [00:00<00:00, 6729.77it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from 

In [10]:
from torch.optim import AdamW
from torch.utils.data import DataLoader

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=8)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

optimizer = AdamW(model.parameters(), lr=2e-5)

In [11]:
epochs = 2

for epoch in range(epochs):
    model.train()
    total_loss = 0

    for batch in train_loader:
        optimizer.zero_grad()

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(
            input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss
        total_loss += loss.item()

        loss.backward()
        optimizer.step()

    print(f"Epoch {epoch+1}, Loss: {total_loss/len(train_loader)}")

Epoch 1, Loss: 0.005828443077474367
Epoch 2, Loss: 7.2161722171586e-05


In [12]:
model.save_pretrained("bert_finetuned_model")
tokenizer.save_pretrained("bert_finetuned_model")

print("BERT model saved!")

Writing model shards: 100%|███████████████████████| 1/1 [00:00<00:00,  4.92it/s]

BERT model saved!


In [ ]:
predictions = []

model.eval()

with torch.no_grad():
    for batch in test_loader:
        outputs = model(
            batch["input_ids"].to(device),
            attention_mask=batch["attention_mask"].to(device)
        )

        preds = torch.argmax(outputs.logits, dim=1)
        predictions.extend(preds.cpu().numpy())

In [ ]:
print(len(y_test))
print("hello")
print(len(predictions))

In [ ]:
predictions = np.array(predictions)
y_test = np.array(test_labels)   # IMPORTANT FIX

In [ ]:
from sklearn.metrics import classification_report, accuracy_score

print("Accuracy:", accuracy_score(y_test, predictions))
print(classification_report(y_test, predictions))

In [ ]:
## BERT Model Summary

- Used pretrained BERT base uncased model
- Fine-tuned on IMDb dataset
- Uses attention mechanism for context understanding

## Key advantages:
- Understands context better than LSTM
- Handles long-range dependencies
- Achieves higher accuracy

## Limitation:
- Requires more computation time
- Slower training compared to LSTM